In [ ]:
!pip install -q peft
!pip install -U "torchao>=0.16.0"
!pip install -q -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.4 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 99.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 39.9 MB/s eta 0:00:00


In [ ]:
import os
import time
import torch
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

TRAIN_PATH = "/kaggle/input/competitions/dlp-nppe-1-t-22026/train.csv" #"/kaggle/input/datasets/somsubhrad/nppe1-dummy-check/train_sample.csv"
TEST_PATH = "/kaggle/input/competitions/dlp-nppe-1-t-22026/test.csv" #"/kaggle/input/datasets/somsubhrad/nppe1-dummy-check/test_sample.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

unique_labels = sorted(train_df["label"].unique())

label2id_int = {int(l): i for i, l in enumerate(unique_labels)}

label2id = {str(int(l)): i for i, l in enumerate(unique_labels)}
id2label = {i: str(int(l)) for i, l in enumerate(unique_labels)}

train_df["label_id"] = train_df["label"].map(label2id_int).astype(int)

num_labels = len(unique_labels)
print(f"Loaded {len(train_df)} training rows. Found {num_labels} distinct categories.")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


cuda
Loaded 50840 training rows. Found 30 distinct categories.


In [ ]:
from huggingface_hub import login
HF_TOKEN = "" #removed before submission

login(token=HF_TOKEN)

In [ ]:
def build_datasets(tokenizer, max_length):
    def tok_fn(examples):
        return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=max_length)

    hf_train = Dataset.from_pandas(train_df[["text", "label_id"]].rename(columns={"label_id": "labels"}))
    hf_test = Dataset.from_pandas(test_df[["text"]])

    tok_train = hf_train.map(tok_fn, batched=True, remove_columns=["text"])
    tok_test = hf_test.map(tok_fn, batched=True, remove_columns=["text"])

    keep_cols = [c for c in ["input_ids", "attention_mask", "token_type_ids"] if c in tok_train.column_names]
    tok_train.set_format(type="torch", columns=keep_cols + ["labels"])
    tok_test.set_format(type="torch", columns=keep_cols)
    return tok_train, tok_test


def train_backbone(model_name, target_modules, output_dir, epochs=2, lr=5e-4,
                    batch_size=16, grad_accum=1, max_length=512, model_kwargs=None):
    model_kwargs = model_kwargs or {}
    stage_start = time.time()
    print(f"\n===== [{time.strftime('%H:%M:%S')}] Starting {model_name} "
          f"(epochs={epochs}, max_length={max_length}, batch={batch_size}, grad_accum={grad_accum}) =====")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tok_train, tok_test = build_datasets(tokenizer, max_length=max_length)

    base_model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=num_labels, id2label=id2label, label2id=label2id, **model_kwargs
    )

    peft_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=16,
        lora_alpha=32,
        lora_dropout=0.1,
        target_modules=target_modules,
    )
    model = get_peft_model(base_model, peft_config)
    model.print_trainable_parameters()
    model.to(device)

    training_args = TrainingArguments(
        output_dir=output_dir,
        learning_rate=lr,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum,
        num_train_epochs=epochs,
        weight_decay=0.01,
        logging_steps=50,
        logging_first_step=True,
        disable_tqdm=False,
        save_strategy="epoch",
        fp16=torch.cuda.is_available(),
        report_to="none",
        label_names=["labels"],
        remove_unused_columns=False,
    )

    class SafeTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
            labels = inputs.pop("labels").long()
            outputs = model(**inputs)
            logits = outputs.logits if hasattr(outputs, "logits") else outputs[0]
            loss = torch.nn.functional.cross_entropy(logits.view(-1, num_labels), labels.view(-1))
            return (loss, outputs) if return_outputs else loss

    trainer = SafeTrainer(model=model, args=training_args, train_dataset=tok_train)
    trainer.train()

    elapsed = time.time() - stage_start
    print(f"===== [{time.strftime('%H:%M:%S')}] Finished {model_name} "
          f"in {elapsed/60:.1f} min =====")

    preds = trainer.predict(tok_test)
    logits = torch.tensor(preds.predictions)
    probs = torch.softmax(logits, dim=1).numpy()
    return probs

In [ ]:
# ignoring the legal bert for this submission
'''# Run Legal-BERT
legalbert_probs = train_backbone(
    model_name="nlpaueb/legal-bert-base-uncased",
    target_modules=["query", "value"],
    output_dir="./legal_bert_lora_nppe",
    epochs=2,
    lr=5e-4,
    batch_size=16,
    grad_accum=1,
    max_length=512,
)'''

'# Run Legal-BERT\nlegalbert_probs = train_backbone(\n    model_name="nlpaueb/legal-bert-base-uncased",\n    target_modules=["query", "value"],\n    output_dir="./legal_bert_lora_nppe",\n    epochs=2,\n    lr=5e-4,\n    batch_size=16,\n    grad_accum=1,\n    max_length=512,\n)'

In [ ]:
#Run ModernBERT
modernbert_probs = train_backbone(
    model_name="answerdotai/ModernBERT-base",
    target_modules=["Wqkv", "Wo"],
    output_dir="./modernbert_lora_nppe",
    epochs=3,
    lr=5e-4,
    batch_size=8,
    grad_accum=2,          # effective batch size 16
    max_length=1024,       # can support 8192 tokens
    model_kwargs={"attn_implementation": "sdpa"},
)


===== [17:02:12] Starting answerdotai/ModernBERT-base (epochs=3, max_length=1024, batch=8, grad_accum=2) =====


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 2,320,926 || all params: 151,948,860 || trainable%: 1.5274


[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,7.885531
50,5.482773
100,3.832804
150,3.178473
200,2.927490
250,2.667527
300,2.725785
350,2.487122
400,2.470328
450,2.533230


===== [00:39:15] Finished answerdotai/ModernBERT-base in 457.1 min =====


In [ ]:
# Ensembling them
# W_LEGAL = 0.5
# Only using ModernBERT, ignoring legalbert
W_MODERN = 1 #0.5

ensemble_probs = W_MODERN * modernbert_probs #+ W_LEGAL * legalbert_probs
predicted_class_ids = np.argmax(ensemble_probs, axis=1)
predicted_labels = [id2label[c] for c in predicted_class_ids]

submission = pd.DataFrame({
    "ID": test_df["id"],
    "label": predicted_labels,
})
submission.to_csv("/kaggle/working/submission.csv", index=False)
print("Saved submission.csv")
submission.head()

Saved submission.csv


,ID,label
0,0,7
1,1,2
2,2,3
3,3,11
4,4,0
